<div dir="rtl">

# Juggernaut XL v9

همه‌ی سلول‌ها را به ترتیب اجرا کنید (Run all). لینک پنل در خروجی سلول آخر چاپ می‌شود.

توکن هاگینگ‌فیس (اختیاری، برای سرعت دانلود): از بخش Secrets کولب با نام HF_TOKEN بسازید.

</div>

In [ ]:
!nvidia-smi -L

In [ ]:
!pip install -q gradio
!pip install -q git+https://github.com/huggingface/diffusers.git

In [ ]:
import os
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"
try:
    from google.colab import userdata
    _tok = userdata.get("HF_TOKEN")
    os.environ["HF_TOKEN"] = _tok
    from huggingface_hub import login
    login(token=_tok)
    print("توکن تنظیم شد — دانلود سریع‌تر انجام می‌شود")
except Exception:
    print("توکنی در Secrets نیست — بدون توکن ادامه می‌دهیم")

In [ ]:
from huggingface_hub import snapshot_download
# دانلود فقط فایل‌های لازم مدل (۶٫۹۴ گیگ)
snapshot_download("RunDiffusion/Juggernaut-XL-v9", allow_patterns=["*.fp16.safetensors", "*.json", "*.txt", "*.model"])
print("دانلود کامل شد")

In [ ]:
%%writefile panel_app.py
import random
import gradio as gr
import numpy as np
import torch
from diffusers import (
    StableDiffusionXLPipeline,
    StableDiffusionXLImg2ImgPipeline,
    StableDiffusionXLInpaintPipeline,
    EulerAncestralDiscreteScheduler,
    EulerDiscreteScheduler,
    DPMSolverMultistepScheduler,
    DPMSolverSinglestepScheduler,
    UniPCMultistepScheduler,
)
from PIL import Image

MAX_SEED = np.iinfo(np.int32).max
MODEL = "RunDiffusion/Juggernaut-XL-v9"
DEFAULT_PROMPT = "Cinematic mid shot photo of an astronaut walking through a neon-lit Tokyo alley at night, hyperdetailed photography, skin details, shallow depth of field"
DEFAULT_NEG = ""
DEF_STEPS = 35
DEF_CFG = 5.0
DEF_SAMPLER = "DPM++ 2M Karras"

# مدل به‌طور کامل روی GPU بارگذاری می‌شود و هر سه پایپ‌لاین از همین وزن‌ها استفاده می‌کنند
pipe = StableDiffusionXLPipeline.from_pretrained(
    MODEL, torch_dtype=torch.float16, variant="fp16", use_safetensors=True,
).to("cuda")
img2img_pipe = StableDiffusionXLImg2ImgPipeline(**pipe.components)
inpaint_pipe = StableDiffusionXLInpaintPipeline(**pipe.components)
ALL_PIPES = [pipe, img2img_pipe, inpaint_pipe]
print("Pipelines ready (txt2img + img2img + inpaint, shared weights)", flush=True)

SCHEDULERS = {
    "DPM++ 2M Karras": lambda cfg: DPMSolverMultistepScheduler.from_config(cfg, use_karras_sigmas=True, algorithm_type="dpmsolver++"),
    "DPM++ SDE Karras": lambda cfg: DPMSolverSinglestepScheduler.from_config(cfg, use_karras_sigmas=True, algorithm_type="dpmsolver++"),
    "Euler a": lambda cfg: EulerAncestralDiscreteScheduler.from_config(cfg),
    "Euler": lambda cfg: EulerDiscreteScheduler.from_config(cfg),
    "UniPC": lambda cfg: UniPCMultistepScheduler.from_config(cfg),
}

def set_scheduler(name):
    for p in ALL_PIPES:
        p.scheduler = SCHEDULERS[name](p.scheduler.config)

RESOLUTIONS = [
    "832 x 1216 (Portrait 2:3)", "1216 x 832 (Landscape 3:2)",
    "1024 x 1024 (Square)", "896 x 1152 (Portrait 3:4)",
    "1152 x 896 (Landscape 4:3)", "768 x 1344 (Tall 9:16)",
    "1344 x 768 (Wide 16:9)", "1024 x 1536 (Tall 2:3)", "1536 x 1024 (Wide 3:2)",
]

def _gen(seed):
    return torch.Generator(device="cuda").manual_seed(int(seed))

def hires_pass(image, prompt, negative, scale, denoise, hsteps, sampler, seed):
    if scale <= 1.0:
        return image
    w, h = image.size
    big = image.resize((int(w * scale), int(h * scale)), Image.LANCZOS)
    set_scheduler(sampler)
    return img2img_pipe(
        prompt=prompt, negative_prompt=negative, image=big,
        strength=float(denoise), num_inference_steps=int(hsteps),
        guidance_scale=5.0, generator=_gen(seed),
    ).images[0]

def txt2img(prompt, prompt2, negative, resolution, steps, guidance, sampler,
            count, seed, randomize_seed, hires_on, hires_scale, hires_denoise, hires_steps):
    set_scheduler(sampler)
    if randomize_seed:
        seed = random.randint(0, MAX_SEED)
    w, h = [int(x) for x in resolution.split("(")[0].strip().split(" x ")]
    out = pipe(
        prompt=prompt, prompt_2=prompt2 or None,
        negative_prompt=negative,
        width=w, height=h,
        num_inference_steps=int(steps), guidance_scale=float(guidance),
        num_images_per_prompt=int(count),
        generator=_gen(seed),
    ).images
    if hires_on:
        out = [hires_pass(im, prompt, negative, float(hires_scale), float(hires_denoise), int(hires_steps), sampler, seed) for im in out]
    return out, int(seed)

def img2img(image, prompt, negative, strength, steps, guidance, sampler, seed, randomize_seed):
    if image is None:
        raise gr.Error("Upload an input image first")
    set_scheduler(sampler)
    if randomize_seed:
        seed = random.randint(0, MAX_SEED)
    out = img2img_pipe(
        prompt=prompt, negative_prompt=negative, image=image,
        strength=float(strength),
        num_inference_steps=int(steps), guidance_scale=float(guidance),
        generator=_gen(seed),
    ).images[0]
    return out, int(seed)

def inpaint(editor, prompt, negative, strength, steps, guidance, sampler, seed, randomize_seed):
    if editor is None or editor.get("background") is None:
        raise gr.Error("Upload an image and paint the mask first")
    set_scheduler(sampler)
    if randomize_seed:
        seed = random.randint(0, MAX_SEED)
    bg = editor["background"].convert("RGB")
    layers = editor.get("layers") or []
    mask = layers[0].split()[3].convert("L") if layers else Image.new("L", bg.size, 0)
    out = inpaint_pipe(
        prompt=prompt, negative_prompt=negative, image=bg, mask_image=mask,
        strength=float(strength),
        num_inference_steps=int(steps), guidance_scale=float(guidance),
        generator=_gen(seed),
    ).images[0]
    return out, int(seed)

def load_lora(repo, weight_name, scale):
    if not (repo or "").strip():
        raise gr.Error("Give a Hugging Face repo id or a local file path")
    kw = {"weight_name": weight_name.strip()} if (weight_name or "").strip() else {}
    for p in ALL_PIPES:
        p.load_lora_weights(repo.strip(), **kw)
        p.set_adapters(["default_0"], adapter_weights=[float(scale)])
    return f"LoRA loaded: {repo} @ {scale}"

def unload_lora():
    for p in ALL_PIPES:
        p.unload_lora_weights()
    return "LoRA unloaded"

with gr.Blocks(title="Juggernaut XL v9") as demo:
    gr.Markdown("# Juggernaut XL v9")
    with gr.Tabs():
        with gr.Tab("Text to Image"):
            with gr.Row():
                with gr.Column(scale=3):
                    t_prompt = gr.Textbox(label="Prompt", value=DEFAULT_PROMPT, lines=3)
                    t_prompt2 = gr.Textbox(label="Prompt 2 (second encoder, optional)", lines=1)
                    t_negative = gr.Textbox(label="Negative prompt", value=DEFAULT_NEG, lines=2)
                    t_resolution = gr.Dropdown(RESOLUTIONS, value=RESOLUTIONS[0], label="Resolution")
                    with gr.Row():
                        t_steps = gr.Slider(1, 80, value=DEF_STEPS, step=1, label="Steps")
                        t_guidance = gr.Slider(1.0, 15.0, value=DEF_CFG, step=0.1, label="Guidance (CFG)")
                        t_sampler = gr.Dropdown(list(SCHEDULERS), value=DEF_SAMPLER, label="Sampler")
                    with gr.Row():
                        t_count = gr.Slider(1, 4, value=1, step=1, label="Images")
                        t_seed = gr.Number(value=42, precision=0, label="Seed")
                        t_random = gr.Checkbox(value=True, label="Randomize seed")
                    with gr.Accordion("Hires fix (upscale + refine pass)", open=False):
                        h_on = gr.Checkbox(value=False, label="Enable hires fix")
                        h_scale = gr.Slider(1.1, 1.5, value=1.3, step=0.05, label="Upscale by")
                        h_denoise = gr.Slider(0.05, 0.5, value=0.25, step=0.05, label="Denoising strength")
                        h_steps = gr.Slider(5, 40, value=20, step=1, label="Hires steps")
                    t_run = gr.Button("Generate", variant="primary")
                with gr.Column(scale=2):
                    t_out = gr.Gallery(label="Result", columns=2)
                    t_used = gr.Number(label="Seed used", precision=0)
            t_run.click(txt2img,
                [t_prompt, t_prompt2, t_negative, t_resolution, t_steps, t_guidance, t_sampler, t_count, t_seed, t_random, h_on, h_scale, h_denoise, h_steps],
                [t_out, t_used])
        with gr.Tab("Image to Image"):
            with gr.Row():
                with gr.Column(scale=3):
                    i_image = gr.Image(label="Input image", type="pil")
                    i_prompt = gr.Textbox(label="Prompt", lines=2)
                    i_negative = gr.Textbox(label="Negative prompt", value=DEFAULT_NEG, lines=2)
                    with gr.Row():
                        i_strength = gr.Slider(0.05, 1.0, value=0.5, step=0.05, label="Strength (how much to change)")
                        i_steps = gr.Slider(1, 80, value=DEF_STEPS, step=1, label="Steps")
                        i_guidance = gr.Slider(1.0, 15.0, value=DEF_CFG, step=0.1, label="Guidance (CFG)")
                        i_sampler = gr.Dropdown(list(SCHEDULERS), value=DEF_SAMPLER, label="Sampler")
                    with gr.Row():
                        i_seed = gr.Number(value=42, precision=0, label="Seed")
                        i_random = gr.Checkbox(value=True, label="Randomize seed")
                    i_run = gr.Button("Generate", variant="primary")
                with gr.Column(scale=2):
                    i_out = gr.Image(label="Result", type="pil")
                    i_used = gr.Number(label="Seed used", precision=0)
            i_run.click(img2img,
                [i_image, i_prompt, i_negative, i_strength, i_steps, i_guidance, i_sampler, i_seed, i_random],
                [i_out, i_used])
        with gr.Tab("Inpaint"):
            with gr.Row():
                with gr.Column(scale=3):
                    n_editor = gr.ImageEditor(label="Image + paint the mask", type="pil",
                                              brush=gr.Brush(colors=["#FFFFFF"], default_size=40))
                    n_prompt = gr.Textbox(label="Prompt (what fills the mask)", lines=2)
                    n_negative = gr.Textbox(label="Negative prompt", value=DEFAULT_NEG, lines=2)
                    with gr.Row():
                        n_strength = gr.Slider(0.05, 1.0, value=0.85, step=0.05, label="Strength")
                        n_steps = gr.Slider(1, 80, value=DEF_STEPS, step=1, label="Steps")
                        n_guidance = gr.Slider(1.0, 15.0, value=DEF_CFG, step=0.1, label="Guidance (CFG)")
                        n_sampler = gr.Dropdown(list(SCHEDULERS), value=DEF_SAMPLER, label="Sampler")
                    with gr.Row():
                        n_seed = gr.Number(value=42, precision=0, label="Seed")
                        n_random = gr.Checkbox(value=True, label="Randomize seed")
                    n_run = gr.Button("Inpaint", variant="primary")
                with gr.Column(scale=2):
                    n_out = gr.Image(label="Result", type="pil")
                    n_used = gr.Number(label="Seed used", precision=0)
            n_run.click(inpaint,
                [n_editor, n_prompt, n_negative, n_strength, n_steps, n_guidance, n_sampler, n_seed, n_random],
                [n_out, n_used])
        with gr.Tab("LoRA"):
            gr.Markdown("Load any SDXL LoRA from Hugging Face (repo id, and the .safetensors file name if the repo has several). Applies to all tabs.")
            l_repo = gr.Textbox(label="LoRA repo / path", placeholder="e.g. someone/sdxl-lora")
            l_file = gr.Textbox(label="File name (optional)", placeholder="e.g. lora.safetensors")
            l_scale = gr.Slider(0.0, 1.5, value=0.8, step=0.05, label="LoRA weight")
            with gr.Row():
                l_load = gr.Button("Load LoRA", variant="primary")
                l_unload = gr.Button("Unload LoRA")
            l_status = gr.Textbox(label="Status", interactive=False)
            l_load.click(load_lora, [l_repo, l_file, l_scale], l_status)
            l_unload.click(unload_lora, [], l_status)

demo.launch(share=True)


In [ ]:
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -u panel_app.py